In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import streamlit as st
st.set_page_config(page_title="Manhwa", layout="centered")

In [ ]:


##data cleaning and preprocessing
@st.cache_data
def prepare_data(file): 
    df = pd.read_csv(file)
    df['chapters']=df['chapters'].fillna(0) 
    df['genres']=df['genres'].fillna('') 
    df['popularity']=df['popularity'].fillna(0) 
    df['score']=df['score'].fillna(0) 
    df['mean_score']=df['mean_score'].fillna(0)
    df['description']=df['description'].fillna('')
    df['soup']=df['genres']+ " "+df['description']
    scaler = MinMaxScaler()
    df[['popularity', 'score']] = scaler.fit_transform(df[['popularity', 'score']])
    return df

In [ ]:
@st.cache_data
def load_similarity_matrix():
    return np.load('cosine_sim.npy')

In [ ]:
def recommender(title, df, cosine_sim, w_sim, w_score, w_pop, status_filter):
    matches = df[df['title'].str.lower() == title.lower()]
    if matches.empty:
        return None
    
    target_idx = matches.index[0]
    sim_scores = list(enumerate(cosine_sim[target_idx]))
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:50] 
    
    candidate_keys = [pair[0] for pair in sorted_scores]
    sim_values = [pair[1] for pair in sorted_scores]
    candidate_df = df.iloc[candidate_keys].copy()
    candidate_df['sim_score'] = sim_values
    if status_filter != "All":  # Apply status filter 
        candidate_df = candidate_df[candidate_df['status'] == status_filter]
    candidate_df['final_score'] = (         #recommendation weights
        (w_sim * candidate_df['sim_score']) + 
        (w_score * candidate_df['score']) + 
        (w_pop * candidate_df['popularity'])
    )
    
    return candidate_df.sort_values(by='final_score', ascending=False).head(10)

In [ ]:
#title 
st.title("Manhwa Recommendation")

# Sidebar Controls
st.sidebar.header("Algorithm Controls")

w_sim = st.sidebar.slider("Plot & Genre Similarity Weight", 0.0, 1.0, 0.5, 0.05)
w_score = st.sidebar.slider("Rating Weight", 0.0, 1.0, 0.3, 0.05)
w_pop = st.sidebar.slider("Popularity Weight", 0.0, 1.0, 0.2, 0.05)

st.sidebar.divider()
status_filter = st.sidebar.selectbox("Filter by Status", ["All", "FINISHED", "RELEASING"])

try :
    df = prepare_data('manhwa_industry_evolution_2026.csv')
    cosine_sim = load_similarity_matrix()
    titles_list = sorted(df['title'].astype(str).tolist())
    selected_title = st.selectbox("Select or search for a Manhwa title:", 
        options=titles_list,
        index=None,
        placeholder="Solo Leveling ..."
    )
    if selected_title:
        results = recommender(selected_title, df, cosine_sim, w_sim, w_score, w_pop, status_filter)
        target_info = df[df['title'] == selected_title].iloc[0]
        st.info(f"**Selected:** {target_info['title']} | **Genres:** {target_info['genres']} | **Rating:** {target_info['mean_score']}/100")
        if results is not None:
            st.subheader(f"Recommended titles for '{selected_title}':")
            for idx, row in results.iterrows():
                with st.container():
                    col1, col2, col3 = st.columns([3, 1, 1])
                    with col1:
                        st.markdown(f"### {row['title']}")
                        st.caption(f"**Genres:** {row['genres']} | **Status:** {row['status']}")
                    with col2:
                        st.metric("Plot Match", f"{round(row['sim_score'] * 100, 1)}%")
                    with col3:
                        st.metric("Score", f"{int(row['mean_score'])}/100")
                    
                    with st.expander("Read Synopsis"):
                        st.write(row['description'])
                    st.divider()
        else:
            st.error("No results found.")
except FileNotFoundError:
    st.error("File not found. Please download the file and try again.")

